In [1]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import make_column_transformer
from sklearn.svm import SVR
from sklearn.pipeline import make_pipeline
import warnings
warnings.filterwarnings('ignore')
from sklearn.preprocessing import StandardScaler
from sklearn.compose import TransformedTargetRegressor

In [2]:
df = pd.read_csv('final.csv')

In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 679 entries, 0 to 678
Data columns (total 6 columns):
 #   Column      Non-Null Count  Dtype
---  ------      --------------  -----
 0   name        679 non-null    str  
 1   company     679 non-null    str  
 2   year        679 non-null    int64
 3   Price       679 non-null    int64
 4   kms_driven  679 non-null    int64
 5   fuel_type   679 non-null    str  
dtypes: int64(3), str(3)
memory usage: 48.7 KB


In [4]:
# Divide dataset into X and y

X = df[['name', 'company', 'year', 'kms_driven', 'fuel_type']]
y = df[['Price']]

In [5]:
X["name"].unique()

<ArrowStringArray>
[        'Santro Xing XO',         'Jeep CL550 MDI',        'Grand i10 Magna',
 'EcoSport Titanium 1.5L',                   'Figo',                    'Eon',
 'EcoSport Ambiente 1.5L',        'Suzuki Alto K10',      'Fabia Classic 1.2',
    'Suzuki Stingray VXi',
 ...
                  'Manza',                'Etios G',                 'Qualis',
              'Quanto C4',     'i20 Select Variant',         'City VX Petrol',
                   'Getz',                  'Fabia',          'Indica V2 DLE',
         'Zest XM Diesel']
Length: 368, dtype: str

In [6]:
X["company"].unique()

<ArrowStringArray>
[   'Hyundai',   'Mahindra',       'Ford',     'Maruti',      'Skoda',
       'Audi',     'Toyota',    'Renault',      'Honda',     'Datsun',
       'Tata', 'Volkswagen',  'Chevrolet',        'BMW',     'Nissan',
  'Hindustan',       'Fiat',      'Force',   'Mercedes', 'Mitsubishi',
       'Jeep']
Length: 21, dtype: str

In [7]:
X["fuel_type"].unique()

<ArrowStringArray>
['Petrol', 'Diesel', 'LPG']
Length: 3, dtype: str

In [8]:
#Lets build pipeline

#Create obj for OneHotEncoder
ohe = OneHotEncoder()
ohe.fit(X[['company', 'name', 'fuel_type']])
# ohe.categories_
ct = make_column_transformer((OneHotEncoder(categories = ohe.categories_), ['company', 'name', 'fuel_type']), remainder = 'passthrough')

model = TransformedTargetRegressor(
    regressor=SVR(kernel='rbf', C=100, gamma='scale'),
    transformer=StandardScaler()
)

# pass input as ct to model
#train to pipeline
pipe = make_pipeline(ct, model)
pipe

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('columntransformer', ...), ('transformedtargetregressor', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('onehotencoder', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'passthrough'
,"sparse_threshold sparse_thresh

In [9]:
# #Lets divide data into train and test
# from sklearn.model_selection import train_test_split
# from sklearn.metrics import r2_score
# # scores = []

# # # To get the best accuracy we need to change random_score value to get best accuracy score. using for loop we can achieve it.

# # for i in range(0, 101):
# X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.1, random_state = 8)
# #Lets train pipe
# pipe.fit(X_train, y_train)
# #Lets test and find accuracy

#     # scores.append(score)

# y_pred = pipe.predict(X_test)
# y_pred
# y_pred = pd.DataFrame(data = y_pred, columns = ['Prediction'])
# result = pd.concat([y_test.reset_index(drop = True), y_pred], axis = 1)
# score = r2_score(result['Price'], result['Prediction'])
# score
#Lets divide data into train and test
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score
scores = []

# To get the best accuracy we need to change random_score value to get best accuracy score. using for loop we can achieve it.

for i in range(0, 101):
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.1, random_state = i)
    #Lets train pipe
    pipe.fit(X_train, y_train)
    #Lets test and find accuracy
    y_pred = pipe.predict(X_test)
    y_pred = pd.DataFrame(data = y_pred, columns = ['Prediction'])
    result = pd.concat([y_test.reset_index(drop = True), y_pred], axis = 1)
    score = r2_score(result['Price'], result['Prediction'])
    scores.append(score)

# scores

In [10]:
best_index = np.argmax(scores)
# print(scores[best_index])
print(f"{best_index} : {scores[best_index]}" )

68 : 0.16815996626654317


In [11]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.1, random_state = best_index)
#Lets train pipe with best_index
pipe.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('columntransformer', ...), ('transformedtargetregressor', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](5,)","['name','company','year','kms_driven','fuel_type']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,5
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('onehotencoder', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsi

In [15]:
data = [['Ford', 'Figo', 2015, 70000, 'Petrol']]
columns = ['company', 'name', 'year', 'kms_driven', 'fuel_type']
myinput = pd.DataFrame(data = data, columns = columns)
result = pipe.predict(myinput)
print(result)
print(type(result))
print(result.shape)
print("Predicted price is:", round(result[0][0]))

[[193633.54442206]]
<class 'numpy.ndarray'>
(1, 1)
Predicted price is: 193634


In [14]:
import pickle as pkl

pkl.dump(pipe, open("model3.pkl", "wb"))